In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.rwf2000 import RWF2000Dataset
from src.baseline_cnn_lstm import BaselineCNNLSTM
from src.baseline_cnn_lstm_2 import BaselineCNNLSTM2
from src.config import DATASET_ROOT
from src.config import CHECKPOINT_DIR
from tqdm.notebook import tqdm

In [8]:
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")
print(device)

cuda:5


In [10]:
# baseline model 1
hyperparameters = {
    "num_frames": 32, 
    "batch_size": 4,
    "hidden_size": 256,
    "learning_rate": 5e-5,
    "epochs": 15,
    "dropout": 0.4,
    "augment": True,
}

model = BaselineCNNLSTM(
    hidden_size=hyperparameters["hidden_size"],
    num_layers=1,
    num_classes=2,
    dropout=hyperparameters["dropout"],
    freeze_cnn=True
).to(device)

In [39]:
# baseline model 2

hyperparameters = {
    "num_frames": 32, 
    "batch_size": 2,
    "hidden_channels": 128,
    "learning_rate": 5e-5,
    "epochs": 50,
    "dropout": 0.4,
    "augment": True,
}

model = BaselineCNNLSTM2(
    hidden_channels=hyperparameters["hidden_channels"],
    num_classes=2,
    dropout=hyperparameters["dropout"],
    freeze_cnn=True
).to(device)


In [11]:
# dataset, loaders, model, optimizer, criterion

train_dataset = RWF2000Dataset(DATASET_ROOT, split="train", num_frames=hyperparameters["num_frames"], augment=hyperparameters["augment"])
val_dataset = RWF2000Dataset(DATASET_ROOT, split="val", num_frames=hyperparameters["num_frames"], augment=hyperparameters["augment"])

train_loader = DataLoader(
    train_dataset,
    batch_size=hyperparameters["batch_size"],
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=hyperparameters["batch_size"],
    shuffle=False,
    num_workers=4
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=hyperparameters["learning_rate"]
)

In [12]:
# training function

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    
    for videos, labels in pbar:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * videos.size(0)
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct/total:.4f}"
        )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [13]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for videos, labels in pbar:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * videos.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{correct/total:.4f}"
            )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [14]:
model_name = "v1.pt"
save_dir = CHECKPOINT_DIR / "baseline_cnn_SepConvLSTM" / model_name

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

num_epochs = hyperparameters["epochs"]

best_val_acc = 0.0
epochs_without_improvement = 0
early_stopping_patience = 10

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)


    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)    
    history["val_acc"].append(val_acc)
    
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "config": hyperparameters,
            "best_val_acc": best_val_acc
        }

        torch.save(checkpoint, save_dir)
        print(f"New best model saved. Val Acc: {best_val_acc:.4f}")

    else:
        epochs_without_improvement += 1
        print(
            f"No improvement for {epochs_without_improvement}/"
            f"{early_stopping_patience} epochs"
        )
        
    if epochs_without_improvement >= early_stopping_patience:
        print("Early stopping triggered.")
        break

print('Training complete normally.')


Epoch 1/50
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]

Validation:   0%|          | 0/200 [00:00<?, ?it/s]

Train Loss: 0.6884 | Train Acc: 0.5456 | Val Loss: 0.6740 | Val Acc: 0.7025
New best model saved. Val Acc: 0.7025

Epoch 2/50
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]

...

Train Loss: 0.6123 | Train Acc: 0.6719 | Val Loss: 0.5674 | Val Acc: 0.6825
No improvement for 1/10 epochs

Epoch 8/50
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]

Validation:   0%|          | 0/200 [00:00<?, ?it/s]

Train Loss: 0.5860 | Train Acc: 0.6881 | Val Loss: 0.5422 | Val Acc: 0.7050
No improvement for 2/10 epochs

Epoch 9/50
--------------------------------------------------


Training:   0%|          | 0/800 [00:00<?, ?it/s]